In [2]:
import cv2
import numpy as np
import os
import argparse
import matplotlib.pyplot as plt
from datetime import datetime

def create_parser():
    parser = argparse.ArgumentParser(description='Video Annotation Tool')
    parser.add_argument('--input', '-i', type=str, required=True, help='Path to input video file')
    parser.add_argument('--output', '-o', type=str, help='Path to output annotated video file')
    parser.add_argument('--fps', type=int, default=30, help='FPS for output video')
    parser.add_argument('--text', '-t', type=str, default='', help='Text to annotate video with')
    parser.add_argument('--timestamp', action='store_true', help='Add timestamp to video')
    parser.add_argument('--box', action='store_true', help='Add bounding box around text')
    parser.add_argument('--position', type=str, default='bottom_left',
                       choices=['top_left', 'top_right', 'bottom_left', 'bottom_right', 'center'],
                       help='Position of the annotation')
    return parser

def calculate_position(frame_width, frame_height, text_size, position):
    text_width, text_height = text_size
    padding = 10

    if position == 'top_left':
        return (padding, text_height + padding)
    elif position == 'top_right':
        return (frame_width - text_width - padding, text_height + padding)
    elif position == 'bottom_left':
        return (padding, frame_height - padding)
    elif position == 'bottom_right':
        return (frame_width - text_width - padding, frame_height - padding)
    elif position == 'center':
        return (frame_width // 2 - text_width // 2, frame_height // 2 + text_height // 2)

    # Default to bottom left
    return (padding, frame_height - padding)

def annotate_video(input_path, output_path=None, fps=30, text='', add_timestamp=False,
                  add_box=False, position='bottom_left'):
    # Open the video file
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_path}")
        return False

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_fps = cap.get(cv2.CAP_PROP_FPS)

    # Set output path if not specified
    if output_path is None:
        filename, ext = os.path.splitext(input_path)
        output_path = f"{filename}_annotated{ext}"

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Set text properties
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    font_color = (255, 255, 255)  # White
    line_type = 2

    # Process each frame
    frame_number = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Prepare text to display
        display_text = text
        if add_timestamp:
            timestamp = frame_number / original_fps
            minutes = int(timestamp // 60)
            seconds = int(timestamp % 60)
            milliseconds = int((timestamp % 1) * 1000)
            time_text = f"{minutes:02d}:{seconds:02d}.{milliseconds:03d}"

            if display_text:
                display_text += f" | {time_text}"
            else:
                display_text = time_text

        if display_text:
            # Get text size for positioning
            (text_width, text_height), baseline = cv2.getTextSize(
                display_text, font, font_scale, line_type)

            # Calculate position
            text_position = calculate_position(width, height, (text_width, text_height), position)

            # Add background box if requested
            if add_box:
                box_color = (0, 0, 0)  # Black
                box_coords = (
                    text_position[0] - 5,
                    text_position[1] + 5,
                    text_position[0] + text_width + 5,
                    text_position[1] - text_height - 5
                )
                cv2.rectangle(frame,
                              (box_coords[0], box_coords[3]),
                              (box_coords[2], box_coords[1]),
                              box_color, -1)  # -1 fills the rectangle

            # Add text to frame
            cv2.putText(frame, display_text, text_position, font, font_scale,
                       font_color, line_type)

        # Write frame to output video
        out.write(frame)
        frame_number += 1

        # Show progress
        if frame_number % 100 == 0:
            print(f"Processing frames: {frame_number}/{frame_count} ({frame_number/frame_count*100:.1f}%)")

    # Release resources
    cap.release()
    out.release()
    print(f"Annotation complete. Output saved to {output_path}")
    return True

def display_sample_frame(input_path, text='', add_timestamp=False, add_box=False, position='bottom_left'):
    # Open the video file
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_path}")
        return False

    # Read the middle frame
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count // 2)
    ret, frame = cap.read()
    if not ret:
        print("Error: Could not read frame from video")
        return False

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    original_fps = cap.get(cv2.CAP_PROP_FPS)

    # Set text properties
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    font_color = (255, 255, 255)  # White
    line_type = 2

    # Prepare text to display
    display_text = text
    if add_timestamp:
        timestamp = frame_count // 2 / original_fps
        minutes = int(timestamp // 60)
        seconds = int(timestamp % 60)
        milliseconds = int((timestamp % 1) * 1000)
        time_text = f"{minutes:02d}:{seconds:02d}.{milliseconds:03d}"

        if display_text:
            display_text += f" | {time_text}"
        else:
            display_text = time_text

    if display_text:
        # Get text size for positioning
        (text_width, text_height), baseline = cv2.getTextSize(
            display_text, font, font_scale, line_type)

        # Calculate position
        text_position = calculate_position(width, height, (text_width, text_height), position)

        # Add background box if requested
        if add_box:
            box_color = (0, 0, 0)  # Black
            box_coords = (
                text_position[0] - 5,
                text_position[1] + 5,
                text_position[0] + text_width + 5,
                text_position[1] - text_height - 5
            )
            cv2.rectangle(frame,
                         (box_coords[0], box_coords[3]),
                         (box_coords[2], box_coords[1]),
                         box_color, -1)  # -1 fills the rectangle

        # Add text to frame
        cv2.putText(frame, display_text, text_position, font, font)